# Belle II Continuum Suppression

Binary signal-versus-continuum classification using the instructor's Belle II CSV. The private data is not committed.

## 1. Setup and reproducibility

In [ ]:
from pathlib import Path
import importlib.util
import sys
import pandas as pd
from IPython.display import Image

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.baselines import evaluate_baselines
from src.config import (
    DB_PATH, RANDOM_STATE, RAW_DATA_DIR, TEST_SIZE, VALIDATION_SIZE
)
from src.data_io import (
    assign_split, audit_frame, load_raw_to_db, prepare_professor_binary
)
from src.db import read_table, table_exists, update_cell, write_table
from src.eda import (class_balance, feature_columns, feature_correlations,
                     feature_separation, plot_class_balance,
                     plot_correlation_heatmap, plot_feature_distributions,
                     training_events)
from src.cnn import fit_cnn_validation
from src.features import analyze_feature_reduction, plot_pca_variance
from src.models import (fit_ankit_validation, fit_koustav_mlp_validation,
                        fit_vishruti_validation)

print(f"Project root: {PROJECT_ROOT}")
print(
    f"Random state: {RANDOM_STATE}; split fractions: "
    f"train={1 - VALIDATION_SIZE - TEST_SIZE:.0%}, "
    f"validation={VALIDATION_SIZE:.0%}, test={TEST_SIZE:.0%}"
)

## 2. Instructor dataset

Use `data_hep - data_hep.csv` only. Its process flags are converted to the binary target below.

The original CSV remains private and unchanged.

## 3. Locate the instructor CSV

In [ ]:
raw_csv_path = RAW_DATA_DIR / "data_hep - data_hep.csv"
if not raw_csv_path.is_file():
    raw_csv_path = None

if raw_csv_path is None:
    print("Waiting for data/raw/data_hep - data_hep.csv.")
else:
    print(f"Dataset found: {raw_csv_path.name}")

## 4. Store the untouched CSV in SQLite

In [ ]:
raw_events = None
if raw_csv_path is not None:
    if table_exists("raw_events", DB_PATH):
        raw_events = read_table("raw_events", DB_PATH)
        print("Read the existing immutable raw_events table.")
    else:
        raw_events = load_raw_to_db(raw_csv_path, DB_PATH)
        print("Wrote the immutable raw_events table.")
    print(f"Loaded {len(raw_events):,} rows and {raw_events.shape[1]} columns.")
else:
    print("Ingestion skipped until exactly one source CSV is available.")

## 5. Audit the raw data

In [ ]:
raw_audit = audit_frame(raw_events) if raw_events is not None else None
if raw_audit is not None:
    print(f"Rows: {raw_audit['rows']:,}; columns: {raw_audit['columns']}")
    print(f"Duplicate rows: {raw_audit['duplicate_rows']:,}")
    print(f"Constant columns: {raw_audit['constant_columns']}")
    print(f"Six-way type counts: {raw_events['type'].value_counts().sort_index().to_dict()}")
    print(f"Missing cells: {sum(raw_audit['missing_counts'].values()):,}")
    print(f"Non-finite cells: {sum(raw_audit['non_finite_counts'].values()):,}")
    display(raw_events.head())
else:
    print("Audit skipped because no dataset has been loaded.")

## 6. Prepare the binary label

`prepare_professor_binary` applies the instructor's process mapping: 0 = charged B pair, 1 = neutral B pair, 2 = $c\bar c$, 3 = $u\bar u$, 4 = $d\bar d$, 5 = $s\bar s$. It combines 0/1 as signal (`target=1`) and 2/3/4/5 as continuum background (`target=0`). The original `type` and exported row number are not model inputs.

In [ ]:
prepared_events = prepare_professor_binary(raw_events) if raw_events is not None else None
if prepared_events is not None:
    print(f"Rows retained: {len(prepared_events):,}/{len(raw_events):,}")
    print("Excluded from prepared data: Unnamed: 0 and six-way type")
    display(prepared_events['target'].value_counts().sort_index().rename('rows'))
else:
    print("Binary preparation waits for the professor CSV.")

## 7. Store the 70/15/15 train/validation/test split

Split by row position, so duplicate pandas index labels cannot overlap assignments.

In [ ]:
events_with_split = None
if prepared_events is not None:
    if table_exists("cleaned_events", DB_PATH):
        events_with_split = read_table("cleaned_events", DB_PATH)
        print("Read persisted cleaned_events; did not re-split.")
    else:
        events_with_split = assign_split(
            prepared_events, "target",
            validation_size=VALIDATION_SIZE, test_size=TEST_SIZE,
        )
        write_table(events_with_split, "cleaned_events", db_path=DB_PATH)
        print("Wrote cleaned_events with one persisted split.")
    display(events_with_split.groupby(["split", "target"]).size().rename("rows"))
else:
    print("Split assignment waits for professor data.")

## 8. Training-only EDA and class balance

Explore training rows only; validation and test remain held out. Raw `type` and `Unnamed: 0` columns are rejected. Counts are unweighted.

In [ ]:
train_events = training_events(events_with_split) if events_with_split is not None else None
if train_events is not None:
    physics_features = feature_columns(train_events)
    print(f'Training rows: {len(train_events):,}; features: {len(physics_features)}')
    balance = class_balance(train_events)
    display(balance)
    display(plot_class_balance(balance))
else:
    print('EDA waits for the instructor dataset.')

## 9. Feature distributions and separation

Plot representative event-shape variables; rank all physics features by single-feature ROC-AUC. This ranking is not a final model result.

In [ ]:
separation = None
if train_events is not None:
    display(plot_feature_distributions(train_events,
           ['R2', 'thrust', 'thrustBm', 'foxWolframR2']))
    separation = feature_separation(train_events)
    display(separation.head(15))
else:
    print('Feature EDA waits for training rows.')

## 10. Correlations and R2 physics check

Check whether signal has lower R2 than continuum; investigate any reversal before modeling.

In [ ]:
if train_events is not None:
    display(plot_correlation_heatmap(feature_correlations(train_events)))
    r2_summary = separation.loc[separation['feature'] == 'R2'].iloc[0]
    print(f"R2 median: signal={r2_summary['signal_median']:.4f}, background={r2_summary['background_median']:.4f}")
    print('Expected R2 direction?', r2_summary['signal_median'] < r2_summary['background_median'])
else:
    print('Physics check waits for training rows.')

## 11. SQLite read, write, and modify

Store and update only the derived training EDA summary; raw and cleaned events stay unchanged.

In [ ]:
if separation is not None:
    eda_summary = separation.assign(status='computed')
    write_table(eda_summary, 'eda_feature_summary', if_exists='replace', db_path=DB_PATH)
    top_feature = eda_summary.iloc[0]['feature']
    changed = update_cell('eda_feature_summary', 'status', 'candidate_for_review',
                          'feature', top_feature, db_path=DB_PATH)
    print(f'Updated {changed} derived row(s): {top_feature}')
    display(read_table('eda_feature_summary', DB_PATH).head(10))
else:
    print('Derived SQLite summary waits for EDA.')

## 12. Training-defined baselines

Compare a training-majority prediction and a training-selected R2 cut on validation. Test rows remain untouched.

In [ ]:
baseline_definition, baseline_validation_results = (None, None)
if events_with_split is not None:
    baseline_definition, baseline_validation_results = evaluate_baselines(events_with_split)
    print(f'Training-selected R2 cut: {baseline_definition.r2_cut:.6f}')
    display(baseline_validation_results)
else:
    print('Baselines wait for the instructor data.')

## 13. Ankit's full-feature models

Train Logistic Regression and Decision Tree on stored training rows; show provisional validation metrics only. The final test split stays untouched.

In [ ]:
ankit_fitted_models, ankit_validation_results = (None, None)
if events_with_split is not None:
    ankit_fitted_models, ankit_validation_results = fit_ankit_validation(events_with_split)
    display(ankit_validation_results)
    print('Provisional validation metrics; final test rows are untouched.')
else:
    print('Modeling waits for the instructor data.')

## 14. Vishruti's full-feature models

Train a scaled Linear SVM and histogram Gradient Boosting BDT; compare all implemented methods on validation only.

In [ ]:
vishruti_fitted_models, vishruti_validation_results = (None, None)
if events_with_split is not None:
    vishruti_fitted_models, vishruti_validation_results = fit_vishruti_validation(events_with_split)
    provisional_validation_table = pd.concat(
        [baseline_validation_results, ankit_validation_results, vishruti_validation_results],
        ignore_index=True,
    )
    display(provisional_validation_table.sort_values('roc_auc', ascending=False))
    print('Validation-only comparison; final test rows remain untouched.')
else:
    print('Vishruti models wait for the instructor data.')

## 15. Training-only feature reduction

Prune correlations above 0.95, inspect standardized PCA at 95% cumulative variance, rank tree importance, and freeze a 20-feature Logistic Regression RFE subset. Validation and test values do not influence this analysis.

In [ ]:
reduction = None
if events_with_split is not None:
    reduction = analyze_feature_reduction(events_with_split)
    print(f'Correlation pruning: {len(reduction.correlation_kept)} kept, {len(reduction.correlation_dropped)} dropped')
    print(f'PCA components for at least 95% training variance: {reduction.pca_components_95}')
    display(reduction.tree_importance.head(15))
    display(reduction.rfe_ranking.head(25))
    display(plot_pca_variance(reduction))
else:
    print('Feature reduction waits for the instructor data.')

## 16. Koustav's MLP

Compare the same scaled feed-forward neural network on all physics features and the frozen RFE subset. Its early-stopping partition is drawn only from training rows.

In [ ]:
mlp_full_model, mlp_full_results = (None, None)
mlp_reduced_model, mlp_reduced_results = (None, None)
if reduction is not None:
    mlp_full_model, mlp_full_results = fit_koustav_mlp_validation(events_with_split)
    mlp_reduced_model, mlp_reduced_results = fit_koustav_mlp_validation(
        events_with_split, reduction.selected_features
    )
    display(pd.concat([mlp_full_results, mlp_reduced_results], ignore_index=True))
else:
    print('MLP validation waits for the training-only reduction result.')

## 17. Experimental physics-grouped CNN

Convolution is used only for ordered CLEO-cone and KSFW-moment families; remaining scalar measurements use a dense branch. PyTorch is optional so the core notebook remains runnable without the large dependency.

In [ ]:
cnn_model, cnn_validation_results = (None, None)
if events_with_split is None:
    print('CNN validation waits for the instructor data.')
elif importlib.util.find_spec('torch') is None:
    print('CNN skipped: install the optional dependency with python -m pip install -r requirements-cnn.txt')
else:
    cnn_model, cnn_validation_results = fit_cnn_validation(events_with_split)
    display(cnn_validation_results)

## 18. Provisional validation comparison

This table supports model and feature-set selection. It is not a final result because the held-out test split remains unopened.

In [ ]:
if mlp_full_results is not None:
    validation_parts = [provisional_validation_table, mlp_full_results, mlp_reduced_results]
    if cnn_validation_results is not None:
        validation_parts.append(cnn_validation_results)
    koustav_validation_table = pd.concat(validation_parts, ignore_index=True)
    display(koustav_validation_table.sort_values('roc_auc', ascending=False))
    print('Freeze the final protocol before evaluating the test split.')
else:
    print('Combined comparison waits for Koustav model results.')

## 19. Evaluation design: two honest comparison studies

The project has two result sets because the models were evaluated at different stages, not because weaker results are being hidden.

**Study A ? frozen held-out test.** The six required classifier families, two simple baselines, and the predeclared reduced MLP were finalized using training/validation evidence. Their settings were then refit on all 60,015 development rows and evaluated once on 10,591 untouched test rows. This table supports the primary project conclusion.

**Study B ? development-only extension comparison.** Random Forest, the full-versus-reduced BDT sensitivity study, and the deeper dense network were proposed after Study A was complete. They therefore use the same five stratified outer folds over the 60,015 train+validation rows, producing one out-of-fold probability per row while excluding every original test row. These four rows can be compared directly with one another because their assessment rows and outer folds match.

The notebook places both studies together for readability, but does not create a single cross-study leaderboard: a test score and a development-OOF score answer related but statistically different questions.

## 20. Study A ? frozen held-out-test comparison

The notebook reads the committed artifacts rather than rerunning the one-shot evaluator. This protects the test set from repeated model selection and keeps the report reproducible.

In [ ]:
final_metrics_path = PROJECT_ROOT / 'results/tables/final_test_metrics.csv'
final_roc_path = PROJECT_ROOT / 'results/figures/final_test_roc.png'
final_confusion_path = PROJECT_ROOT / 'results/figures/final_test_confusion_matrices.png'
if final_metrics_path.is_file():
    final_metrics = pd.read_csv(final_metrics_path)
    display(final_metrics.sort_values('roc_auc', ascending=False))
    display(Image(filename=str(final_roc_path)))
    display(Image(filename=str(final_confusion_path)))
else:
    print('Frozen Study A artifacts are unavailable; do not recreate them casually.')

### Study A finding

The histogram Gradient Boosting BDT is the strongest frozen-test model: accuracy 0.8784, F1 0.8815, and ROC-AUC 0.9457. The full MLP follows at ROC-AUC 0.9416. Its RFE-20 version reaches 0.9399, a decrease of only 0.0017 while using 20 instead of 59 measurements; feature reduction therefore preserves comparable MLP discrimination but does not improve it. The experimental physics-grouped CNN reaches 0.9273 and remains below both MLP variants and the BDT.

This is the headline comparison because all listed configurations were fixed before the test was opened.

## 21. Study B ? additional development-OOF experiments

The extension models answer two follow-up questions: whether other nonlinear models work well with the frozen RFE-20 measurements, and how much the best tree family loses when reduced from 59 to 20 inputs. All rows below use five-fold development-only out-of-fold evaluation and exclude the original test set.

In [ ]:
extension_paths = [
    PROJECT_ROOT / 'results/tables/posthoc_reduced_random_forest_cv.csv',
    PROJECT_ROOT / 'results/tables/posthoc_bdt_feature_comparison_cv.csv',
    PROJECT_ROOT / 'results/tables/posthoc_dense_rfe20_cv.csv',
]
missing_extension_paths = [path for path in extension_paths if not path.is_file()]
if not missing_extension_paths:
    extension_metrics = pd.concat(
        [pd.read_csv(path) for path in extension_paths], ignore_index=True
    ).sort_values('roc_auc', ascending=False)
    display(extension_metrics[[
        'model', 'feature_set', 'evaluation', 'rows', 'features',
        'accuracy', 'precision', 'recall', 'f1', 'roc_auc'
    ]])
else:
    print('Missing Study B artifacts:', [str(path) for path in missing_extension_paths])

In [ ]:
extension_figures = [
    PROJECT_ROOT / 'results/figures/posthoc_bdt_feature_comparison.png',
    PROJECT_ROOT / 'results/figures/posthoc_dense_rfe20_diagnostics.png',
    PROJECT_ROOT / 'results/figures/posthoc_reduced_random_forest_diagnostics.png',
]
for figure_path in extension_figures:
    if figure_path.is_file():
        display(Image(filename=str(figure_path)))

### Study B comparison

Within the common development-OOF protocol, the full BDT ranks first at ROC-AUC 0.9481. Among reduced 20-feature extensions, the deep dense network reaches 0.9432, the reduced BDT 0.9419, and Random Forest 0.9332. The deep network also has the highest reduced-model recall, 0.9012, at lower precision of 0.8549.

The matched BDT comparison isolates the cost of reduction: moving from 59 to 20 inputs lowers ROC-AUC by 0.0062, accuracy by 0.0083, and F1 by 0.0079. The reduced representation retains most discrimination, but the full BDT still benefits from information or interactions outside RFE-20.

The deep network selected epochs `(44, 58, 59, 60, 56)`. Two folds near the fixed 60-epoch ceiling indicate possible incomplete convergence; the budget was not increased after seeing the result.

## 22. Study C ? seed robustness of the representative best models

The full BDT and deep dense RFE-20 network represent the strongest full and reduced conclusions. Ten fixed seeds (42, 123, 2026, 7, 19, 73, 314, 2718, 4096, and 10007) each create a complete five-fold development-only OOF prediction set. The table reports seed-level metrics and mean, sample standard deviation, minimum, and maximum across the ten complete repetitions. Original test rows remain excluded. Run `python -m scripts.run_seed_robustness` separately.

In [ ]:
seed_rows_path = PROJECT_ROOT / 'results/tables/seed_robustness_per_seed.csv'
seed_summary_path = PROJECT_ROOT / 'results/tables/seed_robustness_summary.csv'
seed_figure_path = PROJECT_ROOT / 'results/figures/seed_robustness.png'
if seed_rows_path.is_file() and seed_summary_path.is_file():
    display(pd.read_csv(seed_rows_path))
    display(pd.read_csv(seed_summary_path))
    display(Image(filename=str(seed_figure_path)))
else:
    print('Study C has not run. Use python -m scripts.run_seed_robustness; test rows remain excluded.')

### Study C result

Across ten complete development-OOF repetitions, the full BDT reaches ROC-AUC **0.948093 ? 0.000194** (range 0.947593?0.948268) and F1 **0.879696 ? 0.000522**. The deep dense RFE-20 network reaches ROC-AUC **0.943377 ? 0.000460** (range 0.942605?0.944001) and F1 **0.876410 ? 0.000911**.

The BDT leads on every seed: even its minimum ROC-AUC exceeds the dense network's maximum. The dense model varies more, but both standard deviations are small relative to their means. The original conclusion is therefore stable to these ten fold/training seeds: the full BDT is stronger and more stable, while the 20-feature dense network remains competitive with substantially fewer inputs.

## 23. Study D ? full versus reduced Linear Discriminant Analysis

LDA asks whether the two classes can be separated by one covariance-aware linear discriminant. A StandardScaler and automatic covariance shrinkage are fitted inside every fold. Full 59-feature and frozen RFE-20 variants use the same ten seeds and five development-only folds; original test rows remain excluded. Run `python -m scripts.run_lda_study` separately.

In [ ]:
lda_rows_path = PROJECT_ROOT / 'results/tables/lda_per_seed.csv'
lda_summary_path = PROJECT_ROOT / 'results/tables/lda_summary.csv'
lda_diagnostics_path = PROJECT_ROOT / 'results/figures/lda_diagnostics.png'
lda_scores_path = PROJECT_ROOT / 'results/figures/lda_discriminant_scores.png'
if lda_rows_path.is_file() and lda_summary_path.is_file():
    display(pd.read_csv(lda_rows_path))
    display(pd.read_csv(lda_summary_path))
    display(Image(filename=str(lda_diagnostics_path)))
    display(Image(filename=str(lda_scores_path)))
else:
    print('Study D has not run. Use python -m scripts.run_lda_study; original test rows remain excluded.')

### Study D result

Across ten development-OOF repetitions, full LDA reaches accuracy **0.821171 ? 0.000354**, F1 **0.824490 ? 0.000455**, and ROC-AUC **0.899909 ? 0.000061**. RFE-20 LDA reaches accuracy **0.818444 ? 0.000173**, F1 **0.829851 ? 0.000148**, and ROC-AUC **0.894893 ? 0.000090**.

Full LDA is the better ranker and is slightly more accurate. Reduced LDA has much higher recall (0.874056 versus 0.829237) but lower precision (0.789903 versus 0.819799), so its threshold-specific F1 is higher despite its lower ROC-AUC. The discriminant plots show the central 99% of scores to keep rare extreme values from hiding the class overlap; metrics still use every row.

Under the same development-only repeated-OOF setting, both LDA variants remain well below the nonlinear full BDT and deep dense network. This indicates that continuum separation depends materially on nonlinear relationships among event-shape measurements.

## 24. How the studies fit together

Study A identifies the full histogram BDT as the best predeclared model on the untouched final test. Study B quantifies the cost of RFE-20 and identifies the deeper dense network as the strongest reduced extension. Study C shows that the full-BDT advantage persists across ten partition/training seeds. Study D adds an interpretable linear discriminant: LDA is highly stable, but its ROC-AUC near 0.90 shows that one covariance-aware linear axis cannot match the nonlinear models.

These studies are complementary, not one mixed leaderboard. Study A remains the primary held-out claim; Studies B?D are later development-only analyses of reduction, robustness, and linear separability.

## 25. Limitations

- The instructor dataset is simulated analysis-level data, not real detector data.
- The CSV has no confirmed collision/event-group identifier, so splitting is row-stratified and cannot rule out event-level dependence between rows.
- Study A uses one fixed 70/15/15 split, so it does not quantify repeated-split uncertainty.
- RFE-20 was selected using the original training subset and then held fixed; Study B is not nested cross-validation of the complete selection pipeline.
- The CNN representation is physics-motivated but is not detector geometry.
- The deeper dense network uses one fixed architecture and may be constrained by its 60-epoch ceiling.

These limitations bound the claims to reproducible classification on the supplied sample; they do not establish deployment performance on future Belle II data.

## 26. Conclusion

Event-shape measurements provide strong separation between spherical B-pair signal and jet-like continuum background. The full histogram Gradient Boosting BDT gives the strongest primary held-out-test result, ROC-AUC 0.9457. Reducing 59 measurements to the frozen RFE-20 subset preserves most predictive information across MLP and BDT studies, with a small performance cost. The deep dense extension reaches development-OOF ROC-AUC 0.9432, while Random Forest reaches 0.9332.

Ten-seed robustness confirms that the full BDT is stronger and more stable than the compact dense network. LDA adds an interpretable linear benchmark: full LDA averages ROC-AUC 0.899909 and reduced LDA 0.894893, demonstrating useful but clearly incomplete linear separation. The nonlinear advantage is stable rather than an artifact of one seed.

The project demonstrates private-data ingestion, SQLite read/write/modify operations, leakage-aware EDA, feature interpretation and reduction, classical and neural classifiers, full/reduced comparisons, required metrics, diagnostic plots, repeated-seed robustness, interpretable discriminant analysis, reproducible artifacts, and explicit claim boundaries.